# OESD: Oriented Environmental Swiss Dwellings

# Dependencies

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive
import csv
import os
import math

import shapely.wkt as wkt
import shapely.geometry as sg
from shapely.geometry import Polygon, MultiPolygon, box, LineString, Point
from shapely.ops import transform , unary_union, linemerge #cascaded_union
from shapely import affinity

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import ListedColormap, Normalize, BoundaryNorm, TwoSlopeNorm
from matplotlib.collections import PatchCollection
from matplotlib.cm import ScalarMappable, RdYlGn

from skimage import measure
import random
from PIL import Image

from ipywidgets import widgets
import time
import itertools

In [ ]:
#mount google drive
drive.mount ('/content/gdrive', force_remount=True)
folder = "/content/gdrive/MyDrive/O-ESD"
os.chdir(folder)

In [ ]:
#reading OESD
OESD = pd.read_csv('/content/gdrive/MyDrive/O-ESD/OESD.csv')
#drop the unnamed column of OESD
OESD.drop(columns=['Unnamed: 0'], inplace=True)

In [ ]:
import pandas as pd
import os
import random

def save_units_with_zoning_and_entrance(
    OESD,
    unit_size,
    num_units,
    floor_number,
    output_folder='O-ESD_training_set/unit_8_csv_files'
):
    required_zonings = {"zone01", "zone02", "zone03", "zone04"}

    # Create output directory
    os.makedirs(output_folder, exist_ok=True)

    # Filter area rows for target floor
    area_floor = OESD[
        (OESD['entity_type'] == 'area') &
        (OESD['floor_number'] == floor_number)
    ]

    # Count area size per unit
    area_counts = (
        area_floor.groupby('unit_id')
        .size()
        .reset_index(name='area_count')
    )

    # Filter by unit size
    valid_units = area_counts[area_counts['area_count'] == unit_size]['unit_id'].tolist()

    selected_units = []

    for unit_id in valid_units:
        unit_df = OESD[OESD['unit_id'] == unit_id]

        # Check zoning diversity (must include all zones)
        all_zones = set(unit_df['zoning'].dropna().unique())
        if not required_zonings.issubset(all_zones):
            continue

        # Check exactly one ENTRANCE_DOOR
        entrance_count = (unit_df['zoning'] == 'ENTRANCE_DOOR').sum()
        if entrance_count != 1:
            continue

        # 🆕 Check exactly one zone01
        #zone01_count = (unit_df['zoning'] == 'zone01').sum()
        #if zone01_count != 1:
            #continue

        selected_units.append(unit_id)

    if not selected_units:
        print("No units found satisfying all required conditions.")
        return

    # Randomly sample desired number of units
    sampled_units = random.sample(selected_units, min(num_units, len(selected_units)))

    print(f"Selected unit_ids: {sampled_units}")

    # Save each unit as a CSV
    for unit_id in sampled_units:
        unit_df = OESD[OESD['unit_id'] == unit_id]

        # Keep only area on target floor
        area_df = unit_df[
            (unit_df['entity_type'] == 'area') &
            (unit_df['floor_number'] == floor_number)
        ]

        # Add separators and openings
        separator_df = unit_df[unit_df['entity_type'] == 'separator']
        opening_df = unit_df[unit_df['entity_type'] == 'opening']

        combined_df = pd.concat([area_df, separator_df, opening_df], ignore_index=True)

        # Save CSV
        csv_path = os.path.join(output_folder, f"unit_{unit_id}_floor{floor_number}.csv")
        combined_df.to_csv(csv_path, index=False)

        print(f"Saved: {csv_path}")


# Example:
save_units_with_zoning_and_entrance(OESD, unit_size=8, num_units=200, floor_number=21)


In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import os

def plot_unit_zoning(
    csv_path,
    geometry_col="geometry",
    crs="EPSG:4326",
    save_folder=None
):
    df = pd.read_csv(csv_path)

    # Convert to GeoDataFrame
    if geometry_col in df.columns:
        gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df[geometry_col]))
        gdf.set_crs(crs, inplace=True)
    else:
        raise ValueError(f"Geometry column '{geometry_col}' not found.")

    # Setup figure and axis
    fig, ax = plt.subplots(figsize=(10, 10))

    # 🟦 Pastel palette for zoning areas
    pastel_colors = ["#AEC6CF", "#FFB347", "#77DD77", "#FF6961"]
    zoning_categories = ['zone01', 'zone02', 'zone03', 'zone04']
    zoning_colors = {zone: pastel_colors[i] for i, zone in enumerate(zoning_categories)}

    legend_patches = []

    for zone, color in zoning_colors.items():
        subset = gdf[(gdf['entity_type'] == 'area') & (gdf['zoning'] == zone)]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor='black', linewidth=0.8)
            legend_patches.append(mpatches.Patch(color=color, label=f"{zone}"))

    # ▫ Window orientation color palette
    window_palette = {
        'north': "#6495ED",
        'east': "#FFA07A",
        'south': "#90EE90",
        'west': "#FFD700"
    }

    if 'window_orientation' in gdf.columns:
        for direction, color in window_palette.items():
            subset = gdf[
                (gdf['zoning'] == 'WINDOW') &
                (gdf['window_orientation'].str.lower() == direction)
            ]
            if not subset.empty:
                subset.plot(ax=ax, color=color, edgecolor='black', alpha=1.0)
                legend_patches.append(mpatches.Patch(color=color, label=f"Window ({direction.capitalize()})"))

    # ▫ Other special components
    special_styles = {
        'ENTRANCE_DOOR': ('#000000', 1.0, 'Entrance Door'),
        'DOOR': ('#556B2F', 1.0, 'Door'),
        'WALL': ('#C0C0C0', 0.5, 'Wall')
    }

    for zoning_value, (color, alpha, label) in special_styles.items():
        subset = gdf[gdf['zoning'] == zoning_value]
        if not subset.empty:
            subset.plot(ax=ax, color=color, edgecolor='black', alpha=alpha)
            legend_patches.append(mpatches.Patch(color=color, label=label))

    # 🔄 External legend
    ax.legend(
        handles=legend_patches,
        title="Zoning & Elements",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        borderaxespad=0
    )

    # Final plot adjustments
    plot_title = f"Zoning Layout: {os.path.basename(csv_path)}"
    ax.set_title(plot_title)
    ax.set_axis_off()
    plt.tight_layout(rect=[0, 0, 0.8, 1])

    # Save or show
    if save_folder:
        os.makedirs(save_folder, exist_ok=True)
        plot_filename = os.path.splitext(os.path.basename(csv_path))[0] + ".png"
        save_path = os.path.join(save_folder, plot_filename)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"Saved plot to: {save_path}")
    else:
        plt.show()


In [ ]:
def batch_plot_zoning(folder_path, floor_number=None, save_folder="O-ESD_training_set/unit_plots_8"):
    os.makedirs(save_folder, exist_ok=True)

    for file in os.listdir(folder_path):
        if file.endswith('.csv') and (floor_number is None or f"floor{floor_number}" in file):
            csv_path = os.path.join(folder_path, file)
            print(f"Processing: {csv_path}")
            try:
                plot_unit_zoning(csv_path, save_folder=save_folder)
            except Exception as e:
                print(f"Error plotting {file}: {e}")


In [ ]:
batch_plot_zoning('/content/gdrive/MyDrive/swiss_dwellings_v3.0.0/ESD/O-ESD_training_set/unit_8_csv_files')

In [ ]:
#number of files in this directory
import os
folder_path = '/content/gdrive/MyDrive/swiss_dwellings_v3.0.0/ESD/O-ESD_training_set/unit_8_csv_files'
#count the number of files
num_files = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in the folder: {num_files}")
#